# Operadores bit-a-bit e de deslocamento — Tutorial

**Programação C (COMP0512) — UFS — 2026.2**

`chmod 754` dá ao dono do arquivo ler, escrever e executar; ao grupo, ler e executar; aos outros, só
ler. Nove permissões, cada uma um sim ou um não, guardadas num único número. Este tutorial retoma o
programa `permissoes.c` visto hoje — que imprime um modo como o `ls -l` e aplica ordens como `u+x` e
`go-w` — e segue o caminho dos slides: um inteiro como uma fileira de bits, os operadores `&`, `|`, `^`
e `~`, as máscaras, os deslocamentos `<<` e `>>`, a ordem dos bytes na memória e um conjunto inteiro guardado num `unsigned`. Depois
de cada bloco há um exercício com teste automático; no fim vêm as tarefas de modificação e o desafio.

## Objetivos

Ao final deste tutorial você será capaz de:

- Ler e escrever constantes em binário, octal e hexadecimal, e explicar por que `754` não é `0754`
- Usar `&`, `|`, `^` e `~` sabendo o que acontece em cada posição — e distingui-los de `&&`, `||` e `!`
- Ligar, desligar, inverter e testar bits com máscaras, com os parênteses no lugar certo
- Montar máscaras com `1u << k` e extrair campos com `(x >> s) & m`, sem comportamento indefinido
- Explicar a diferença entre little-endian e big-endian e gravar bytes numa ordem fixa com deslocamentos
- Representar um conjunto pequeno num `unsigned` e contar seus elementos com `c &= c - 1`

In [ ]:
# Confira se o gcc está disponível e defina a função que testa os exercícios
!gcc --version | head -1

import subprocess

def testa(comando, esperado, entrada=None, tempo=20):
    '''Roda o comando e compara a saída com o esperado.'''
    try:
        r = subprocess.run(comando, shell=True, capture_output=True, text=True,
                           input=entrada, timeout=tempo)
    except subprocess.TimeoutExpired:
        print(f'Demorou mais de {tempo} s: algum laço está fazendo trabalho à toa?')
        return
    limpa = lambda t: '\n'.join(l.rstrip() for l in t.strip().split('\n'))
    saida, esperado = limpa(r.stdout), limpa(esperado)
    if r.returncode != 0:
        print(f'O programa terminou com erro (status {r.returncode}; '
              '-11 ou 139 = segmentation fault).')
    if r.stderr:
        print(r.stderr[:1500])
    if saida == esperado:
        print('OK!')
    else:
        print('Saída esperada:')
        print(esperado)
        print('Saída obtida:')
        print(saida)

## O programa da aula

Recompile o programa condutor **exatamente como saiu da aula**. Antes de executar, escreva a saída que
você espera: que letras saem para `0644`? O que muda depois de `u+x`, `go+w`, `a-r` e `o+rwx`?

In [ ]:
%%writefile permissoes.c
/* permissoes.c --- le um modo Unix em octal e aplica ordens como as do chmod
   uso: ./permissoes                  (exemplos)
        ./permissoes 754 u-x go+w     (um modo e uma ou mais ordens) */
#include <stdio.h>
#include <stdlib.h>

/* nove bits: rwx do dono (u), rwx do grupo (g), rwx dos outros (o) */
#define R 4u
#define W 2u
#define X 1u

/* imprime os nove bits como o ls -l: rwxr-xr-- */
void mostra(unsigned modo)
{
    const char *letras = "rwx";
    for (int i = 8; i >= 0; i--)
        putchar(modo & (1u << i) ? letras[(8 - i) % 3] : '-');
}

/* aplica uma ordem como "u+x", "go-w" ou "a+r" e devolve o novo modo */
unsigned aplica(unsigned modo, const char *ordem)
{
    unsigned quem = 0, bits = 0, mascara = 0;
    const char *p = ordem;

    for (; *p == 'u' || *p == 'g' || *p == 'o' || *p == 'a'; p++)
        quem |= *p == 'u' ? 4u : *p == 'g' ? 2u : *p == 'o' ? 1u : 7u;
    char op = *p++;
    for (; *p != '\0'; p++)
        bits |= *p == 'r' ? R : *p == 'w' ? W : *p == 'x' ? X : 0u;

    if (quem & 4u) mascara |= bits << 6;   /* dono   */
    if (quem & 2u) mascara |= bits << 3;   /* grupo  */
    if (quem & 1u) mascara |= bits;        /* outros */

    if (op == '+')
        modo |= mascara;
    else if (op == '-')
        modo &= ~mascara;
    else
        fprintf(stderr, "ordem invalida: %s\n", ordem);
    return modo;
}

void mostra_linha(unsigned modo)
{
    printf("%03o ", modo);
    mostra(modo);
    putchar('\n');
}

int main(int argc, char *argv[])
{
    unsigned modo = 0644;
    const char *exemplo[] = {"u+x", "go+w", "a-r", "o+rwx"};
    int n = 4;
    const char **ordens = exemplo;

    if (argc >= 2) {
        modo = strtoul(argv[1], NULL, 8);   /* le em base 8 */
        ordens = (const char **) argv + 2;
        n = argc - 2;
    }
    printf("%-6s", "");
    mostra_linha(modo);
    for (int i = 0; i < n; i++) {
        modo = aplica(modo, ordens[i]);
        printf("%-6s", ordens[i]);
        mostra_linha(modo);
    }
    return 0;
}

In [ ]:
!gcc -Wall -Wextra permissoes.c -o permissoes && ./permissoes
!echo ---
!./permissoes 754 u-x go+w

## 1. Um inteiro é uma fileira de bits

Na base 2, o dígito na posição $i$ (contando da direita, a partir de 0) vale $2^i$. Um `unsigned` de
32 bits é uma fileira de 32 sim/não. Como cada dígito octal corresponde a exatamente 3 bits e cada dígito
hexadecimal a 4, essas bases mostram os bits sem conta nenhuma:

| em C | base | bits |
|---|---|---|
| `0754` | 8 (começa com `0`) | `111 101 100` |
| `0x1EC` | 16 (começa com `0x`) | `1 1110 1100` |
| `492` | 10 | os mesmos — mas quem enxerga? |

As permissões vêm em grupos de três — dono, grupo, outros —, por isso o `chmod` fala octal. E por isso
esquecer o zero é perigoso: `754` também é uma constante válida, só que decimal. Preveja o que
`decimal.c` imprime antes de rodar.

In [ ]:
%%writefile decimal.c
/* decimal.c --- o modo 754 escrito sem o zero na frente */
#include <stdio.h>

void mostra(unsigned modo)
{
    const char *letras = "rwx";
    for (int i = 8; i >= 0; i--)
        putchar(modo & (1u << i) ? letras[(8 - i) % 3] : '-');
}

int main(void)
{
    unsigned modo = 754;   /* rwxr-xr-- ? */
    printf("%o ", modo);
    mostra(modo);
    putchar('\n');
    return 0;
}

In [ ]:
!gcc -Wall -Wextra decimal.c -o decimal && ./decimal

Nenhum aviso. $754 = 512 + 128 + 64 + 32 + 16 + 2$, que em binário é `10 1111 0010` — dez bits, `1362`
em octal. `mostra` só olha os nove de baixo, `011 110 010`: `-wxrw--w-`, permissões que ninguém pediu.

Para mexer em bits, use `unsigned`: num `int`, o bit 31 é o sinal, e deslocar números negativos tem
regras próprias. Num `unsigned`, todo bit é só um bit.

### Exercício 1 — os bits de um número

Escreva `mostra_bits(x, n)`, que imprime os `n` bits mais baixos de `x`, do mais alto para o mais baixo,
com um espaço separando grupos de 4 **contados da direita** (como em `1 1110 1100`). Use `1u << i` para
testar o bit `i`.

In [ ]:
%%writefile ex1.c
#include <stdio.h>

/* Exercicio 1: imprime os n bits mais baixos de x, do bit n-1 ao bit 0,
   com um espaco antes de cada grupo de 4 contados da direita.
   TODO: implemente mostra_bits. */
void mostra_bits(unsigned x, int n)
{
    /* TODO: implemente aqui */
    (void) x;
    (void) n;
}

int main(void)
{
    unsigned teste[] = {0754, 0754, 5, ~0u, (unsigned) -1, 0x80000000u, 0x1EC};
    int largura[] = {9, 12, 4, 32, 8, 32, 9};
    for (int i = 0; i < 7; i++) {
        printf("%2d bits de %#x: ", largura[i], teste[i]);
        mostra_bits(teste[i], largura[i]);
        putchar('\n');
    }
    return 0;
}

In [ ]:
!gcc -g -fsanitize=undefined -Wall -Wextra ex1.c -o ex1
testa('./ex1', '''9 bits de 0x1ec: 1 1110 1100
12 bits de 0x1ec: 0001 1110 1100
 4 bits de 0x5: 0101
32 bits de 0xffffffff: 1111 1111 1111 1111 1111 1111 1111 1111
 8 bits de 0xffffffff: 1111 1111
32 bits de 0x80000000: 1000 0000 0000 0000 0000 0000 0000 0000
 9 bits de 0x1ec: 1 1110 1100''')

## 2. Os operadores `&`, `|`, `^` e `~`

Cada bit do resultado depende só dos bits na **mesma posição** dos operandos:

| `a` | `b` | `a & b` | `a \| b` | `a ^ b` | `~a` |
|---|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 0 | 1 |
| 0 | 1 | 0 | 1 | 1 | 1 |
| 1 | 0 | 0 | 1 | 1 | 0 |
| 1 | 1 | 1 | 1 | 0 | 0 |

`&` só deixa passar onde os dois têm 1; `|` liga onde qualquer um tem 1; `^` marca onde são diferentes;
`~` inverte **todos** os 32 bits — inclusive os que você não está usando. Não confunda com `&&`, `||` e
`!`, que olham o número inteiro e devolvem 0 ou 1: `6 & 1` é 0, mas `6 && 1` é 1. Preveja cada linha
antes de rodar.

In [ ]:
%%writefile operadores.c
#include <stdio.h>

void mostra_bits(unsigned x, int n)
{
    for (int i = n - 1; i >= 0; i--)
        putchar(x & (1u << i) ? '1' : '0');
}

void linha(const char *nome, unsigned x)
{
    printf("%-8s ", nome);
    mostra_bits(x, 9);
    printf("  %04o\n", x & 0777);
}

int main(void)
{
    unsigned a = 0754, m = 0070;
    linha("a", a);
    linha("m", m);
    linha("a & m", a & m);
    linha("a | m", a | m);
    linha("a ^ m", a ^ m);
    linha("~m", ~m);
    printf("~m inteiro: %o\n", ~m);
    printf("6 & 1 = %d, 6 && 1 = %d, ~6 = %d, !6 = %d\n", 6 & 1, 6 && 1, ~6, !6);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra operadores.c -o operadores && ./operadores

`a & m` isolou o grupo; `a | m` ligou o grupo inteiro; `a ^ m` inverteu só o grupo. Nos nove bits,
`~m` é `0707`; inteiro, é `37777777707` — os 23 bits de cima, que eram zero, viraram um.

### Exercício 2 — distância de Hamming

Em quantos bits `a` e `b` diferem? Os bits diferentes são exatamente os 1 de `a ^ b`: implemente `conta`
(quantos bits valem 1) e use-a em `hamming`. Depois o `main` mostra quantos bits mudam de cada número de
0 a 15 para o seguinte — repare no padrão.

In [ ]:
%%writefile ex2.c
#include <stdio.h>

/* Exercicio 2: conta devolve quantos bits de x valem 1;
   hamming devolve em quantos bits a e b diferem.
   TODO: implemente as duas. */
int conta(unsigned x)
{
    /* TODO: implemente aqui */
    (void) x;
    return 0;
}

int hamming(unsigned a, unsigned b)
{
    /* TODO: implemente aqui (use conta e ^) */
    (void) a;
    (void) b;
    return 0;
}

int main(void)
{
    printf("conta(0754) = %d, conta(~0u) = %d, conta(0) = %d\n",
           conta(0754), conta(~0u), conta(0));
    printf("hamming(6, 1) = %d\n", hamming(6, 1));
    printf("hamming(0754, 0644) = %d\n", hamming(0754, 0644));
    printf("hamming(0, ~0u) = %d\n", hamming(0, ~0u));
    printf("de i para i + 1:");
    for (unsigned i = 0; i < 16; i++)
        printf(" %d", hamming(i, i + 1));
    putchar('\n');
    return 0;
}

In [ ]:
!gcc -g -fsanitize=undefined -Wall -Wextra ex2.c -o ex2
testa('./ex2', '''conta(0754) = 6, conta(~0u) = 32, conta(0) = 0
hamming(6, 1) = 3
hamming(0754, 0644) = 2
hamming(0, ~0u) = 32
de i para i + 1: 1 2 1 3 1 2 1 4 1 2 1 3 1 2 1 5''')

## 3. Máscaras

Uma **máscara** tem 1 exatamente nos bits em que se quer mexer. Com o operador certo, muda esses bits e
deixa os outros como estavam:

| quero | escrevo |
|---|---|
| ligar | `x \|= m` |
| desligar | `x &= ~m` |
| inverter | `x ^= m` |
| testar | `(x & m) != 0` |

Em `aplica`, a máscara de `go+w` é montada copiando `W` (`010`) para a posição do grupo (`<< 3`) e para
a dos outros: `000 010 010`, ou `0022`. Depois, `modo |= mascara`.

Testar parece o mais simples dos quatro. `precedencia.c` decide se o dono e o grupo **não** podem
escrever. Preveja as três linhas — e se o `gcc` vai reclamar.

In [ ]:
%%writefile precedencia.c
/* precedencia.c --- quem pode escrever? */
#include <stdio.h>

#define W 2u

/* devolve 1 se a classe (6 = dono, 3 = grupo, 0 = outros) NAO pode escrever */
int nao_escreve(unsigned modo, int classe)
{
    if (modo & W << classe == 0)
        return 1;
    return 0;
}

int main(void)
{
    unsigned modos[] = {0644, 0664, 0444};
    for (int i = 0; i < 3; i++)
        printf("%03o: dono %s, grupo %s\n", modos[i],
               nao_escreve(modos[i], 6) ? "nao escreve" : "escreve",
               nao_escreve(modos[i], 3) ? "nao escreve" : "escreve");
    return 0;
}

In [ ]:
!gcc -Wall -Wextra precedencia.c -o precedencia && ./precedencia

Todo mundo escreve, até em `0444`. Em C, `&`, `^` e `|` têm precedência **menor** que `==` (e `<<` tem
maior). A expressão foi lida como `modo & ((W << classe) == 0)`: a comparação dá 0, e `modo & 0` é sempre
0. O `gcc -Wall` avisou (`suggest parentheses`), mas um aviso não impede o executável. A correção é
`if ((modo & W << classe) == 0)`. Regra: **todo teste de bits leva parênteses em volta do `&`**.

Corrija `precedencia.c` na célula acima e rode de novo: `0644` deve dizer que o grupo não escreve.

### Exercício 3 — de letras para número

Escreva `de_texto`, o inverso de `mostra`: recebe nove caracteres como `"rwxr-x---"` e devolve o modo
(`0750`). A letra na posição `i` da cadeia corresponde ao bit `8 - i`; qualquer caractere diferente de
`-` liga o bit. O `main` testa alguns casos e depois confere a ida e a volta para todos os 512 modos.

In [ ]:
%%writefile ex3.c
#include <stdio.h>

void mostra(unsigned modo, char s[10])
{
    const char *letras = "rwx";
    for (int i = 0; i < 9; i++)
        s[i] = modo & (1u << (8 - i)) ? letras[i % 3] : '-';
    s[9] = '\0';
}

/* Exercicio 3: "rwxr-x---" -> 0750. TODO: implemente de_texto. */
unsigned de_texto(const char *s)
{
    /* TODO: implemente aqui */
    (void) s;
    return 0;
}

int main(void)
{
    const char *casos[] = {"rwxr-x---", "rw-r--r--", "---------", "rwxrwxrwx", "--x-w-r--"};
    for (int i = 0; i < 5; i++)
        printf("%s -> %03o\n", casos[i], de_texto(casos[i]));

    int erros = 0;
    char s[10];
    for (unsigned modo = 0; modo <= 0777; modo++) {
        mostra(modo, s);
        if (de_texto(s) != modo)
            erros++;
    }
    printf("ida e volta: %d erros em 512 modos\n", erros);
    return 0;
}

In [ ]:
!gcc -g -fsanitize=undefined -Wall -Wextra ex3.c -o ex3
testa('./ex3', '''rwxr-x--- -> 750
rw-r--r-- -> 644
--------- -> 000
rwxrwxrwx -> 777
--x-w-r-- -> 124
ida e volta: 0 erros em 512 modos''')

## 4. Deslocamentos

`x << k` empurra os bits de `x` $k$ casas para a esquerda (entram zeros à direita); `x >> k`, para a
direita. Em `unsigned`, sem estouro, `x << k` é $x \cdot 2^k$ e `x >> k` é $\lfloor x / 2^k \rfloor$.
Dois usos aparecem o tempo todo:

- **fabricar** uma máscara: `1u << k` tem um único 1, no bit `k`;
- **extrair** um campo: `(modo >> 3) & 7u` traz as permissões do grupo para os bits 0–2 e corta o resto.

E as bordas? `mascaras.c` calcula `1 << k` para os bits mais altos de um `int` — e um além. Preveja:
o que sai para $k = 31$? E para $k = 32$, um bit que não existe?

In [ ]:
%%writefile mascaras.c
/* mascaras.c --- a mascara do bit k, para k de 28 a 32 */
#include <stdio.h>

int main(void)
{
    for (int k = 28; k <= 32; k++) {
        int m = 1 << k;
        printf("k = %2d: %11d = 0x%08x\n",
               k, m, (unsigned) m);
    }
    return 0;
}

In [ ]:
!gcc -Wall -Wextra mascaras.c -o mascaras && ./mascaras
!echo --- com o sanitizer de comportamento indefinido:
!gcc -g -fsanitize=undefined mascaras.c -o mascaras-ubsan && ./mascaras-ubsan

Nenhum aviso de compilação, e dois problemas:

- `1 << 31`: `1` é `int`, e $2^{31}$ não cabe num `int`. Deslocar um 1 para dentro do bit de sinal é
  **comportamento indefinido**. A correção é `1u << k`.
- `1 << 32`: deslocar por um valor maior ou igual à largura do tipo (ou negativo) é indefinido **mesmo em
  `unsigned`**. O processador costuma usar $k \bmod 32$ — daí o 1 de volta —, mas nada garante.

O `-fsanitize=undefined` aponta esses casos durante a execução. Troque a linha para
`unsigned m = 1u << k;` (e o `%11d` por `%11u`), rode de novo com o sanitizer e veja a queixa mudar para
o `k = 32`.

### Exercício 4 — cores `0xRRGGBB`

Uma cor de 24 bits guarda vermelho, verde e azul, um byte cada: `0xFF8000` é laranja (vermelho 255,
verde 128, azul 0). Implemente `rgb`, que empacota três componentes (use só os 8 bits de baixo de cada
um), as três funções que desempacotam e `cinza`, que troca a cor pela média inteira das três
componentes, repetida nos três bytes.

In [ ]:
%%writefile ex4.c
#include <stdio.h>

/* Exercicio 4: TODO: implemente as cinco funcoes com <<, >> e &. */
unsigned rgb(unsigned r, unsigned g, unsigned b)
{
    /* TODO: implemente aqui */
    (void) r; (void) g; (void) b;
    return 0;
}

unsigned vermelho(unsigned c) { /* TODO */ (void) c; return 0; }
unsigned verde(unsigned c)    { /* TODO */ (void) c; return 0; }
unsigned azul(unsigned c)     { /* TODO */ (void) c; return 0; }

unsigned cinza(unsigned c)
{
    /* TODO: implemente aqui */
    (void) c;
    return 0;
}

int main(void)
{
    printf("rgb(0x12, 0x34, 0x56) = %06x\n", rgb(0x12, 0x34, 0x56));
    printf("rgb(255, 128, 0) = %06x\n", rgb(255, 128, 0));
    printf("rgb(300, 0, 0) = %06x\n", rgb(300, 0, 0));
    unsigned cores[] = {0xFF8000, 0x123456, 0xFFFFFF, 0x0000FF};
    for (int i = 0; i < 4; i++)
        printf("%06x: r = %3u, g = %3u, b = %3u, cinza = %06x\n", cores[i],
               vermelho(cores[i]), verde(cores[i]), azul(cores[i]), cinza(cores[i]));
    return 0;
}

In [ ]:
!gcc -g -fsanitize=undefined -Wall -Wextra ex4.c -o ex4
testa('./ex4', '''rgb(0x12, 0x34, 0x56) = 123456
rgb(255, 128, 0) = ff8000
rgb(300, 0, 0) = 2c0000
ff8000: r = 255, g = 128, b =   0, cinza = 7f7f7f
123456: r =  18, g =  52, b =  86, cinza = 343434
ffffff: r = 255, g = 255, b = 255, cinza = ffffff
0000ff: r =   0, g =   0, b = 255, cinza = 555555''')

## 5. A ordem dos bytes na memória

Um `unsigned` de 32 bits ocupa 4 bytes, em 4 endereços seguidos. Em que ordem? Depende do processador:

- **little-endian** guarda o byte menos significativo no menor endereço — x86 e ARM (no modo usual);
- **big-endian** guarda o mais significativo no menor endereço — a ordem usada pelos protocolos de rede.

Os nomes vêm das *Viagens de Gulliver*, da guerra entre quem quebra o ovo pela ponta menor e quem o quebra
pela maior (Cohen, 1981). `bytes.c` guarda `0x01020304` e olha os bytes um a um por um ponteiro para
`unsigned char`. Preveja: `p[0]` é `01` ou `04`? E `(x >> 24) & 0xFF`? Qual das duas respostas mudaria
num processador big-endian?

In [ ]:
%%writefile bytes.c
/* bytes.c --- os quatro bytes de um unsigned, na ordem da memoria */
#include <stdio.h>

int main(void)
{
    unsigned x = 0x01020304;
    unsigned char *p = (unsigned char *) &x;
    for (int i = 0; i < 4; i++)
        printf("p[%d] = %02x\n", i, p[i]);
    printf("(x >> 24) & 0xFF = %02x\n", (x >> 24) & 0xFFu);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra bytes.c -o bytes && ./bytes

Num processador little-endian, `p[0]` é `04`: a memória guarda os bytes "ao contrário". Num big-endian,
seria `01`. Já `(x >> 24) & 0xFF` dá `01` em qualquer máquina: `>>` é a conta $\lfloor x / 2^{24} \rfloor$,
feita sobre o **valor**, que não sabe como os bytes estão arrumados na memória.

A ordem só importa quando os bytes saem do programa: um `int` gravado com `fwrite` num arquivo binário
(próxima aula) ou enviado pela rede pode ser lido na ordem errada por outra máquina. A saída portátil é
escolher uma ordem para o arquivo e montar os bytes com `>>`, `<<` e `&` — nunca por um ponteiro para o
`int`. Para a rede, `<arpa/inet.h>` tem `htonl` e `ntohl`.

### Exercício 5 — trocando a ordem dos bytes

Implemente `troca_bytes`, que inverte a ordem dos 4 bytes de `x` (`0x01020304` vira `0x04030201`), só
com `<<`, `>>`, `&` e `|`; `le_be`, que monta um `unsigned` a partir de 4 bytes em big-endian (`b[0]` é o
mais significativo); e `eh_little`, que devolve 1 se a máquina é little-endian (olhe o primeiro byte de
um `unsigned` conhecido). O `main` confere que copiar os bytes direto para um `unsigned` dá `le_be` dos
bytes numa máquina big-endian e `troca_bytes(le_be(...))` numa little-endian — a saída é a mesma em
qualquer uma.

In [ ]:
%%writefile ex5.c
#include <stdio.h>
#include <string.h>

/* Exercicio 5: TODO: implemente troca_bytes, le_be e eh_little. */
unsigned troca_bytes(unsigned x)
{
    /* TODO: implemente aqui, com <<, >>, & e | */
    (void) x;
    return 0;
}

unsigned le_be(const unsigned char b[4])
{
    /* TODO: implemente aqui */
    (void) b;
    return 0;
}

int eh_little(void)
{
    /* TODO: implemente aqui */
    return 0;
}

int main(void)
{
    unsigned teste[] = {0x01020304, 0xDEADBEEF, 0x000000FF, 0x80000000};
    for (int i = 0; i < 4; i++)
        printf("troca_bytes(%08x) = %08x\n", teste[i], troca_bytes(teste[i]));
    printf("troca duas vezes volta: %d\n", troca_bytes(troca_bytes(0x12345678)) == 0x12345678);
    unsigned char b[4] = {0xCA, 0xFE, 0x00, 0x2A};
    printf("le_be(ca fe 00 2a) = %08x\n", le_be(b));
    unsigned direto;
    memcpy(&direto, b, 4);
    unsigned esperado = eh_little() ? troca_bytes(le_be(b)) : le_be(b);
    printf("memcpy confere com eh_little: %d\n", direto == esperado);
    return 0;
}

In [ ]:
!gcc -g -fsanitize=undefined -Wall -Wextra ex5.c -o ex5
testa('./ex5', '''troca_bytes(01020304) = 04030201
troca_bytes(deadbeef) = efbeadde
troca_bytes(000000ff) = ff000000
troca_bytes(80000000) = 00000080
troca duas vezes volta: 1
le_be(ca fe 00 2a) = cafe002a
memcpy confere com eh_little: 1''')

## 6. Um conjunto num `unsigned`

Se o bit `i` diz se `i` está no conjunto, um `unsigned` guarda qualquer subconjunto de $\{0, \ldots, 31\}$,
e as operações de conjunto viram uma instrução cada: união `a | b`, interseção `a & b`, diferença
`a & ~b`, diferença simétrica `a ^ b`. Para contar os elementos, `c &= c - 1` apaga o 1 mais à direita —
subtrair 1 transforma esse 1 em 0 e os zeros à direita dele em 1, e o `&` apaga tudo isso de uma vez. O
laço dá uma volta por elemento, não por bit.

In [ ]:
%%writefile conjunto.c
/* conjunto.c --- um conjunto de inteiros de 0 a 31 guardado num unsigned:
   o elemento i esta no conjunto se o bit i vale 1 */
#include <stdio.h>

typedef unsigned Conjunto;

Conjunto insere(Conjunto c, int i)  { return c | 1u << i; }
Conjunto retira(Conjunto c, int i)  { return c & ~(1u << i); }
int pertence(Conjunto c, int i)     { return (c >> i) & 1u; }

/* cada c &= c - 1 apaga o 1 mais a direita */
int conta(Conjunto c)
{
    int n = 0;
    while (c != 0) {
        c &= c - 1;
        n++;
    }
    return n;
}

void imprime(const char *nome, Conjunto c)
{
    printf("%-10s {", nome);
    for (int i = 0; i < 32; i++)
        if (pertence(c, i))
            printf(" %d", i);
    printf(" }  %d elementos\n", conta(c));
}

int main(void)
{
    Conjunto pares = 0, mult3 = 0;
    for (int i = 2; i <= 20; i += 2)
        pares = insere(pares, i);
    for (int i = 3; i <= 20; i += 3)
        mult3 = insere(mult3, i);

    imprime("pares", pares);
    imprime("mult3", mult3);
    imprime("p | m", pares | mult3);
    imprime("p & m", pares & mult3);
    imprime("p & ~m", pares & ~mult3);
    imprime("p ^ m", pares ^ mult3);
    imprime("sem o 6", retira(pares, 6));
    return 0;
}

In [ ]:
!gcc -Wall -Wextra conjunto.c -o conjunto && ./conjunto

Para mais de 32 elementos, use um vetor de `unsigned`: o elemento `i` é o bit `i % 32` da palavra
`i / 32` — ou, com deslocamento e máscara, o bit `i & 31` da palavra `i >> 5`. `crivo.c` conta os primos
até $n$ com o crivo de Eratóstenes de duas formas: um `char` por número e um bit por número. Antes de
rodar: quanta memória cada versão usa para $n = 10^8$? Qual é mais rápida?

In [ ]:
%%writefile crivo.c
/* crivo.c --- quantos primos ate n? O crivo de Eratostenes com um char por
   numero e com um bit por numero.
   uso: ./crivo [n]      (padrao: n = 100000000) */
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

/* composto[i] = 1 se i nao e primo */
long crivo_char(long n)
{
    char *composto = calloc(n + 1, 1);
    long primos = 0;
    for (long i = 2; i <= n; i++) {
        if (composto[i])
            continue;
        primos++;
        for (long j = i * i; j <= n; j += i)
            composto[j] = 1;
    }
    free(composto);
    return primos;
}

/* o bit i % 32 da palavra v[i / 32] vale 1 se i nao e primo */
#define LIGA(v, i)  ((v)[(i) >> 5] |= 1u << ((i) & 31))
#define TESTA(v, i) (((v)[(i) >> 5] >> ((i) & 31)) & 1u)

long crivo_bits(long n)
{
    unsigned *composto = calloc(n / 32 + 1, sizeof(unsigned));
    long primos = 0;
    for (long i = 2; i <= n; i++) {
        if (TESTA(composto, i))
            continue;
        primos++;
        for (long j = i * i; j <= n; j += i)
            LIGA(composto, j);
    }
    free(composto);
    return primos;
}

int main(int argc, char *argv[])
{
    long n = argc > 1 ? atol(argv[1]) : 100000000;
    clock_t t = clock();
    long p = crivo_char(n);
    printf("char: %ld primos, %6.1f MB, %.2f s\n", p, (n + 1) / 1e6,
           (double) (clock() - t) / CLOCKS_PER_SEC);
    t = clock();
    p = crivo_bits(n);
    printf("bits: %ld primos, %6.1f MB, %.2f s\n", p,
           (n / 32 + 1) * sizeof(unsigned) / 1e6,
           (double) (clock() - t) / CLOCKS_PER_SEC);
    return 0;
}

In [ ]:
!gcc -O2 -Wall -Wextra crivo.c -o crivo
for n in [10**6, 10**7, 10**8]:
    print(f'n = {n}:')
    !./crivo {n}

O mesmo número de primos com um oitavo da memória. O tempo fica parecido: a versão com bits faz mais
contas por acesso, mas a memória menor cabe melhor no *cache*. Em problemas que não cabem na memória com
um `char` por item, o bit por item é o que torna o programa possível.

### Exercício 6 — que dia todo mundo pode?

Cada pessoa informa os dias da semana em que pode se reunir como um conjunto: o bit 0 é domingo, o bit 6
é sábado. Implemente `conta` (com `c &= c - 1`), `todos_podem` (os dias em que **todas** as pessoas
podem) e `melhor_dia` (o dia em que **mais** pessoas podem; no empate, o menor dia).

In [ ]:
%%writefile ex6.c
#include <stdio.h>

typedef unsigned Conjunto;
#define DIA(d) (1u << (d))
enum { DOM, SEG, TER, QUA, QUI, SEX, SAB };
const char *nome[] = {"dom", "seg", "ter", "qua", "qui", "sex", "sab"};

/* Exercicio 6: TODO: implemente conta, todos_podem e melhor_dia. */
int conta(Conjunto c)
{
    /* TODO: implemente aqui, com c &= c - 1 */
    (void) c;
    return 0;
}

Conjunto todos_podem(const Conjunto disp[], int n)
{
    /* TODO: implemente aqui */
    (void) disp;
    (void) n;
    return 0;
}

int melhor_dia(const Conjunto disp[], int n)
{
    /* TODO: implemente aqui */
    (void) disp;
    (void) n;
    return 0;
}

void imprime(Conjunto c)
{
    printf("{");
    for (int d = DOM; d <= SAB; d++)
        if (c & DIA(d))
            printf(" %s", nome[d]);
    printf(" } (%d dias)\n", conta(c));
}

int main(void)
{
    Conjunto turma[] = {
        DIA(SEG) | DIA(QUA) | DIA(SEX),
        DIA(SEG) | DIA(TER) | DIA(QUA),
        DIA(QUA) | DIA(QUI) | DIA(SEX) | DIA(SEG),
    };
    Conjunto banda[] = {
        DIA(SAB) | DIA(DOM),
        DIA(SEX) | DIA(SAB),
        DIA(DOM) | DIA(SEG),
        DIA(SEX) | DIA(DOM),
    };
    printf("turma, todos podem: ");
    imprime(todos_podem(turma, 3));
    printf("turma, melhor dia: %s\n", nome[melhor_dia(turma, 3)]);
    printf("banda, todos podem: ");
    imprime(todos_podem(banda, 4));
    printf("banda, melhor dia: %s\n", nome[melhor_dia(banda, 4)]);
    printf("conta(0x7F) = %d, conta(0x80000001) = %d\n", conta(0x7F), conta(0x80000001u));
    return 0;
}

In [ ]:
!gcc -g -fsanitize=undefined -Wall -Wextra ex6.c -o ex6
testa('./ex6', '''turma, todos podem: { seg qua } (2 dias)
turma, melhor dia: seg
banda, todos podem: { } (0 dias)
banda, melhor dia: dom
conta(0x7F) = 7, conta(0x80000001) = 2''')

## Tarefas: altere o programa da aula

Volte à célula de `permissoes.c`, faça cada alteração, reexecute e confira se a saída bate com o que você
previu.

1. **A ordem `=`.** `u=rx` deixa o dono com exatamente `r-x`, qualquer que fosse o antes, sem mexer nas
   outras classes. Implemente em `aplica`: primeiro desligue as três permissões das classes de `quem`
   (que máscara é essa?), depois ligue as pedidas. Teste com `./permissoes 777 u=rx go=` — o que deve
   sair?
2. **A ordem `^`.** Uma ordem que não existe no `chmod`: `go^w` inverte a escrita do grupo e dos outros.
   Que operador resolve em uma linha? Aplicar a mesma ordem duas vezes volta ao modo original — por quê?
3. **`mostra` sem `1u << i`.** Reescreva `mostra` deslocando o próprio modo: teste sempre o bit 8 com
   `modo & 0400` e faça `modo <<= 1` a cada volta. A saída deve ser idêntica.

## Desafio: duas pilhas de mesmo peso

Leia da entrada padrão um número $n \leq 20$ e depois $n$ pesos inteiros. Divida os pesos em duas pilhas
com a menor diferença de peso possível e imprima

```
diferenca: <d>
pilha: <os pesos da primeira pilha, na ordem da entrada>
```

Cada subconjunto dos pesos é um número `mask` de 0 a $2^n - 1$: o bit `i` diz se o peso `i` vai para a
primeira pilha. Percorra todos com `for (unsigned mask = 0; mask < 1u << n; mask++)` e some os pesos dos
bits ligados. Se mais de uma divisão tiver a menor diferença, imprima a de **menor** `mask`.

In [ ]:
%%writefile pilhas.c
#include <stdio.h>
#include <stdlib.h>

#define MAXN 20

/* Desafio: le n e n pesos; imprime a menor diferenca entre duas pilhas e os
   pesos da primeira (a de menor mask entre as melhores).
   TODO: percorra as mascaras de 0 a 2^n - 1. */
int main(void)
{
    int n, peso[MAXN];
    if (scanf("%d", &n) != 1 || n < 1 || n > MAXN)
        return 1;
    for (int i = 0; i < n; i++)
        if (scanf("%d", &peso[i]) != 1)
            return 1;

    /* TODO: sua solucao aqui */

    return 0;
}

In [ ]:
pesos = {
    'o dos slides': ('5\n3 1 4 2 2\n',
        'diferenca: 0\npilha: 3 1 2'),
    'quatro pesos': ('4\n10 20 30 40\n',
        'diferenca: 0\npilha: 20 30'),
    'sem empate': ('3\n1 1 5\n',
        'diferenca: 3\npilha: 1 1'),
    'dois iguais': ('2\n7 7\n',
        'diferenca: 0\npilha: 7'),
    'de 1 a 20': ('20\n1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20\n',
        'diferenca: 0\npilha: 1 2 3 4 5 6 7 8 9 10 11 12 13 14'),
    'potencias': ('12\n1 3 9 27 81 243 729 2187 6561 19683 59049 177147\n',
        'diferenca: 88574\npilha: 1 3 9 27 81 243 729 2187 6561 19683 59049'),
}
!gcc -g -fsanitize=undefined -Wall -Wextra pilhas.c -o pilhas
for nome, (entrada, esperado) in pesos.items():
    print(nome + ':', end=' ')
    testa('./pilhas', esperado, entrada=entrada)

Depois de passar no teste, experimente: quantas máscaras o programa percorre para $n = 20$? E para
$n = 30$ — quanto tempo levaria? Por que `1u << n` deixa de funcionar para $n = 32$? Este é o problema
da **partição**: ninguém conhece um método que seja muito mais rápido que tentar tudo no pior caso.

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. Os pontos principais desta aula estão em:

- Kernighan e Ritchie (1988), *The C Programming Language*, §2.9 — operadores bit a bit; §2.10 e
  exercício 2-9 — `x &= (x - 1)`; §2.12 — precedência;
- Bryant e O'Hallaron (2015), *Computer Systems: A Programmer's Perspective*, §2.1.1–2.1.9 —
  hexadecimal, ordem dos bytes (§2.1.3), operações bit a bit, lógicas e deslocamentos;
- King (2008), *C Programming: A Modern Approach*, §20.1 — operadores bit a bit;
- Deitel e Deitel (2011), *C: como programar*, cap. 10 — manipulação de bits;
- ISO/IEC 9899:2018, §6.4.4.1 — constantes octais e hexadecimais; §6.5.7 — deslocamentos e os casos
  indefinidos;
- Warren (2013), *Hacker's Delight*, §2-1 — os bits mais à direita; §7-1 — invertendo bits e bytes; cap. 5 — contando bits;
- Knuth (2011), *The Art of Computer Programming*, vol. 4A, §7.1.3 — truques bit a bit; §7.2.1.1 —
  gerando todas as $n$-uplas;
- Kerrisk (2010), *The Linux Programming Interface*, §15.4 — permissões de arquivos; §59.2 — a ordem
  dos bytes na rede, `htonl` e `ntohl`;
- Cohen (1981), *On Holy Wars and a Plea for Peace* — a origem dos termos big-endian e little-endian;
- Garey e Johnson (1979), *Computers and Intractability*, SP12 — o problema da partição.